# 70 — P10.8: requisitos clínicos de Susana Torres

Convierte observaciones de entrevista en requisitos trazables. No crea ground truth ni entrena.

In [1]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

from pathlib import Path
from datetime import datetime, timezone
import io, json, os
import pandas as pd

ROOT=Path(os.getenv("PFI_ROOT","/content/drive/MyDrive/PFI_MVP"))
OUT=Path(os.getenv("PFI_P10_8_PREFLIGHT_ROOT",str(ROOT/"results/P10_8_clinical_expansion_preflight")))
if not (OUT/"NOTEBOOK_69_COMPLETE.json").is_file():
    raise FileNotFoundError("Primero debe completarse Notebook 69.")

CSV="""requirementId|requirement|workstream|priority|currentCoverage|trainingRequired|newDatasetPotential|aiModuleImpact|backendImpact|frontendImpact|nextNotebook|status
SUS-01|Taxonomía visual estable por vértebra y estructura|frontend|high|partial|false|false|none|possible_palette_metadata|required|none|product_change_candidate
SUS-02|Medir dirección y magnitud de anterolistesis o retrolistesis|measurement|high|binary_classifier_weak|false|false|required_geometry|required_measurement_contract|required_review_points|74|measurement_research_candidate
SUS-03|No graduar listesis por tercios sin definición clínica congelada|governance|high|not_implemented|false|false|blocked_pending_definition|blocked_pending_definition|do_not_display_grade|73|blocked_pending_clinical_definition
SUS-04|Neuroforamen izquierdo y derecho como estructuras independientes|segmentation|high|classification_without_localization|true|true|required_new_localization|required_structure_contract|required_independent_overlay|71,73|dataset_audit_required
SUS-05|Separar grasa epidural de neuroforámenes y saco tecal|segmentation|medium|not_separated|true|true|required_if_viable|required_if_viable|required_if_viable|71,73|dataset_audit_required
SUS-06|Priorizar diámetros AP y transverso sobre área|measurement|high|partial|false|false|required_measurement_audit|required_contract_order|required_metric_order|74|measurement_audit_required
SUS-07|Auditar localización y características de raíces nerviosas|segmentation|medium|unsupported|true|true|future_new_model|future_new_contract|future_review|71,73|dataset_audit_required
SUS-08|Definir el marco entre disco, platillo y vértebra antes de etiquetar|dataset_audit|medium|endplate_heads_partial|false|false|audit_only|none|possible_quality_indicator|71|definition_required
SUS-09|Formulación multiframe de hernia con sagital, parasagital y axial|classification|high|p10_7_not_product_supported|true|true|future_multiframe_runtime|future_multiseries_traceability|future_multislice_evidence|73,76,77|new_task_definition_required
SUS-10|Máscaras para todos los cortes pertinentes de cada nivel|runtime_and_segmentation|high|representative_slice_only|false|false|required_multiframe_inference|required_multi_asset_contract|required_slice_navigation|75|runtime_change_required
SUS-11|Altura discal anterior, central, posterior y pérdida relativa|measurement|high|narrowing_classifier_only|false|false|required_geometry|required_measurement_contract|required_level_display|74|measurement_research_candidate
SUS-12|Auditar señal T2 sin llamar deshidratación a una regla no validada|classification_and_measurement|medium|pfirrmann_experimental|false|false|signal_audit_possible|future_if_validated|experimental_only|71,73|definition_and_validation_required
SUS-13|Registrar y mostrar sagital T1 y T2 por separado|runtime_and_product|high|training_support_product_pending|false|false|required_modality_registry|required_series_roles|required_t1_t2_selector|76|cross_repo_change_required
SUS-14|Transportar overlays por corte y plano|runtime_and_product|high|partial|false|false|required|required|required|75,76|cross_repo_change_required
SUS-15|Abrir imagen original con overlays desactivados|frontend|high|partial|false|false|none|none|required_default_off|none|frontend_change_candidate
SUS-16|Auditar hipertrofia del ligamento amarillo por nivel y lado|new_finding_candidate|high|unsupported|true|true|future_if_viable|future_if_viable|future_if_viable|71,72,73,77|dataset_audit_required
SUS-17|Auditar hipertrofia o degeneración facetaria|new_finding_candidate|high|unsupported|true|true|future_if_viable|future_if_viable|future_if_viable|71,72,73,77|dataset_audit_required
SUS-18|Mantener infiltración grasa muscular como línea futura|future_scope|low|unsupported|true|true|deferred|deferred|deferred|future|deferred
SUS-19|Conservar geometría DICOM para referencia sagital-axial|geometry_and_product|high|partial|false|false|required_geometry_trace|required_geometry_transport|required_cross_reference|76|regression_and_e2e_required
SUS-20|Resolver nivel lumbar por corte mediante quality.sliceLevels|runtime_and_contract|critical|implemented_p10_6_branch|false|false|must_preserve|must_preserve|must_consume|75,76|must_preserve
"""
df=pd.read_csv(io.StringIO(CSV),sep="|")
for col in ["trainingRequired","newDatasetPotential"]:
    df[col]=df[col].map({"true":True,"false":False})
df["aiModuleChangeLikely"]=~df.aiModuleImpact.isin(["none","deferred","audit_only"])
df["backendChangeLikely"]=~df.backendImpact.isin(["none","deferred","possible_palette_metadata"])
df["frontendChangeLikely"]=~df.frontendImpact.isin(["none","deferred"])

def write_json(path,payload):
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(payload,indent=2,ensure_ascii=False,sort_keys=True)+"\n",encoding="utf-8")
    os.replace(tmp,path)

OUT.mkdir(parents=True,exist_ok=True)
df.to_csv(OUT/"clinical_requirement_registry_v1.csv",index=False)
df[["requirementId","requirement","priority","aiModuleImpact","backendImpact","frontendImpact",
    "nextNotebook","status","aiModuleChangeLikely","backendChangeLikely","frontendChangeLikely"]].to_csv(
    OUT/"implementation_change_watch_v1.csv",index=False)
write_json(OUT/"clinical_requirement_registry_v1.json",{
 "schemaVersion":"pfi.p10-8.clinical-requirement-registry.v1",
 "source":{"interviewee":"Susana Torres","sourceType":"professional_validation_interview",
           "groundTruth":False,"thresholdsValidated":False},
 "generatedAtUtc":datetime.now(timezone.utc).isoformat(),
 "requirements":df.to_dict(orient="records")})
marker={
 "schemaVersion":"pfi.p10-8.notebook-70-complete.v1","status":"NOTEBOOK_70_COMPLETE",
 "generatedAtUtc":datetime.now(timezone.utc).isoformat(),"trainingExecuted":False,
 "clinicalGroundTruthCreated":False,"clinicalThresholdsFrozen":False,
 "requirementCount":len(df),"aiModuleChangeCandidates":int(df.aiModuleChangeLikely.sum()),
 "backendChangeCandidates":int(df.backendChangeLikely.sum()),
 "frontendChangeCandidates":int(df.frontendChangeLikely.sum())}
write_json(OUT/"NOTEBOOK_70_COMPLETE.json",marker)
print(json.dumps(marker,indent=2,ensure_ascii=False))
print("NOTEBOOK_70_COMPLETE")

Mounted at /content/drive
{
  "schemaVersion": "pfi.p10-8.notebook-70-complete.v1",
  "status": "NOTEBOOK_70_COMPLETE",
  "generatedAtUtc": "2026-08-06T23:36:20.178026+00:00",
  "trainingExecuted": false,
  "clinicalGroundTruthCreated": false,
  "clinicalThresholdsFrozen": false,
  "requirementCount": 20,
  "aiModuleChangeCandidates": 16,
  "backendChangeCandidates": 16,
  "frontendChangeCandidates": 19
}
NOTEBOOK_70_COMPLETE
